# 02 Pipeline

Build a governed, quality-checked source-to-target workflow with one visible sequence: **0. Environment → E. Extract → T. Transform → L. Load**.

Each SOURCE or TARGET block is deliberately small and cloneable. FabricOps resolves identity, read scope, target write settings, and successful source completion while the physical reads, business transformation, Guardrails, and physical writes stay visible.

## Tested with FabricOps

The previous baseline of this template was run in Microsoft Fabric with FabricOps v0.2.0 by Voyce on 6 Aug 2026. This redesigned workflow has local structural and public-API compatibility validation only; run it in your configured Fabric workspace before treating it as runtime-validated.

# 0. Environment

Run the shared configuration, then import only the public functions used by this template.

In [ ]:
%run 00_env_config

In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import (
    check_dq,
    check_freshness,
    check_schema,
    profile_and_register_table,
    profile_dataframe,
    read_lakehouse_csv,
    read_lakehouse_excel,
    read_lakehouse_parquet,
    read_lakehouse_table,
    read_pipeline_prep,
    read_warehouse_query,
    read_warehouse_table,
    write_lakehouse_table,
    write_pipeline_prep,
    write_warehouse_table,
)

# E. Extract

## SOURCE 1 — Configure

Declare the physical source once and choose its processing **strategy**. FabricOps turns that strategy into a runtime `read_mode` for this execution:

- strategy: `full_dataset`, `incremental_watermark`, or `incremental_partition`
- read mode: `skip`, `full_dataset`, or `incremental_subset`

The current `read_pipeline_prep()` contract also needs the intended target definition so it can resolve one governed processing definition. Declare it once here; the Load block reuses it.

In [ ]:
# SOURCE 1 — physical identity and engineer-selected strategy
SOURCE_TARGET = "product"
SOURCE_SCHEMA = "dbo"
SOURCE_TABLE_NAME = "student_enrolment"
SOURCE_READ_STRATEGY = "incremental_watermark"
SOURCE_WATERMARK_COLUMN = "modified_datetime"
SOURCE_PARTITION_COLUMN = None

# Publication identity and strategy, declared once for governed preparation
TARGET_TARGET = "unified"
TARGET_SCHEMA = "dbo"
TARGET_TABLE_NAME = "student_enrolment"
TARGET_LOAD_STRATEGY = "scd1"
TARGET_LOAD_PARAMETERS = {"key_columns": ["student_id"]}

# Other valid source configurations:
# SOURCE_READ_STRATEGY = "full_dataset"
# SOURCE_WATERMARK_COLUMN = None
# SOURCE_READ_STRATEGY = "incremental_partition"
# SOURCE_PARTITION_COLUMN = "snapshot_date"

## SOURCE 1 — Prepare and Guard

Preparation derives canonical source and target identity, resolves the configured strategies, and returns the governed physical scope. Schema is checked before business rows are read. Partition observations can also be checked for freshness and source-change Guardrails; watermark state is validated during preparation.

In [ ]:
read_prep = read_pipeline_prep(
    source_table_name=SOURCE_TABLE_NAME,
    target_table_name=TARGET_TABLE_NAME,
    source_read_strategy=SOURCE_READ_STRATEGY,
    source_watermark_column=SOURCE_WATERMARK_COLUMN,
    source_partition_column=SOURCE_PARTITION_COLUMN,
    source_target=SOURCE_TARGET,
    source_schema=SOURCE_SCHEMA,
    target=TARGET_TARGET,
    schema=TARGET_SCHEMA,
    load_strategy=TARGET_LOAD_STRATEGY,
    load_strategy_parameters=TARGET_LOAD_PARAMETERS,
)

schema_result = check_schema(
    SOURCE_TABLE_NAME,
    target=SOURCE_TARGET,
    schema=SOURCE_SCHEMA,
)
pre_read_results = [schema_result]

if read_prep["observation"] is not None:
    pre_read_results.append(check_freshness(read_prep["observation"]))
if read_prep["changes"] is not None:
    pre_read_results.append(read_prep["changes"])

if not all(result["can_continue"] for result in pre_read_results):
    raise RuntimeError("A source Guardrail blocked this run.")

SHOULD_RUN = read_prep["read_mode"] != "skip"
print(f'Runtime read mode: {read_prep["read_mode"]}')

## SOURCE 1 — Read

The reader receives the scope returned by FabricOps. Do not add watermark predicates, partition filters, or checkpoint access here. A `skip` decision bypasses the physical reader and all downstream publication.

In [ ]:
if not SHOULD_RUN:
    source_1_df = None
    print("No source changes detected. Source read and downstream publication are skipped.")
else:
    source_1_df = read_warehouse_table(
        SOURCE_SCHEMA,
        SOURCE_TABLE_NAME,
        target=SOURCE_TARGET,
        spark_session=spark,
        processing_scope=read_prep["scope"],
    )

## SOURCE 1 — DQ and Profile

DQ evaluates the rows selected for this run. A complete source read may replace the canonical registered source profile. An incremental slice is diagnostic only and must not replace the latest complete profile.

In [ ]:
if SHOULD_RUN:
    source_dq_result = check_dq(
        source_1_df,
        SOURCE_TABLE_NAME,
        target=SOURCE_TARGET,
        schema=SOURCE_SCHEMA,
    )
    display(source_dq_result["summary"])
    if not source_dq_result["can_continue"]:
        raise RuntimeError("A source DQ Guardrail blocked this run.")

    if read_prep["read_mode"] == "full_dataset":
        source_profile_df = profile_and_register_table(
            source_1_df,
            profile_role="source",
            target=SOURCE_TARGET,
            schema=SOURCE_SCHEMA,
            table_name=SOURCE_TABLE_NAME,
        )
    elif read_prep["read_mode"] == "incremental_subset":
        source_profile_df = profile_dataframe(source_1_df)

    display(source_profile_df)

### Need another source?

Duplicate the SOURCE block and rename its variables and DataFrame (`source_2_df`, for example). Keep the main runnable pattern one source to one target unless the pipeline has explicitly defined how multiple source `skip` decisions should interact.

### Optional source alternatives

The canonical example reads the configured `product` Warehouse and publishes to the `unified` Lakehouse. For a governed Lakehouse source, replace only the physical reader with:

```python
source_1_df = read_lakehouse_table(
    table_name=SOURCE_TABLE_NAME,
    target=SOURCE_TARGET,
    schema=SOURCE_SCHEMA,
    spark_session=spark,
    processing_scope=read_prep["scope"],
)
```

`read_warehouse_table()` reads one named physical Warehouse table and supports governed watermark or partition pushdown. `read_warehouse_query()` remains the advanced Warehouse option for engineer-authored joins, filters, aggregation, or projection; do not register arbitrary query output as the complete profile of one physical source.

Lakehouse Files are normally a simple full-load staging path: `full_dataset` → transform → `overwrite`. Use `read_lakehouse_csv()`, `read_lakehouse_excel()`, or `read_lakehouse_parquet()` as the physical reader; do not invent incremental file semantics.

In [ ]:
# Optional Lakehouse Files readers — use one in a full_dataset staging pipeline.
# source_1_df = read_lakehouse_csv(
#     "Files/inbound/student_enrolment.csv", target=SOURCE_TARGET,
#     spark_session=spark, header=True, inferSchema=True,
# )
# source_1_df = read_lakehouse_excel(
#     "Files/inbound/student_enrolment.xlsx", target=SOURCE_TARGET,
#     spark_session=spark, sheet_name="enrolment",
# )
# source_1_df = read_lakehouse_parquet(
#     "Files/inbound/student_enrolment.parquet", target=SOURCE_TARGET,
#     spark_session=spark,
# )

# Advanced Warehouse SQL — explicit custom output, not a canonical table profile.
# source_1_df = read_warehouse_query(
#     "SELECT student_id, course_id FROM dbo.student_enrolment WHERE is_active = 1",
#     target=SOURCE_TARGET,
#     spark_session=spark,
# )

# T. Transform

**Business transformation is engineer-owned.** Keep joins, filters, derivations, aggregations, and reshaping visible rather than hiding them in a FabricOps orchestrator.

In [ ]:
if SHOULD_RUN:
    transformed_df = source_1_df.withColumn(
        "ingested_at_utc",
        F.current_timestamp(),
    )
    display(transformed_df)

# Multi-source extension after duplicating the SOURCE block:
# transformed_df = source_1_df.join(source_2_df, on="student_id", how="left")

# L. Load

## TARGET 1 — Prepare and Guard

The target identity and load strategy were declared once for `read_pipeline_prep()`. Here the transformed shape is checked, then `write_pipeline_prep()` prepares the audited DataFrame and exact writer settings. Lakehouse supports `overwrite`, `append`, `scd1`, and `scd2`; configure `key_columns` for SCD strategies and `effective_column` when required by SCD2.

In [ ]:
if SHOULD_RUN:
    target_schema_result = check_schema(
        TARGET_TABLE_NAME,
        target=TARGET_TARGET,
        schema=TARGET_SCHEMA,
        dataframe=transformed_df,
    )
    target_dq_result = check_dq(
        transformed_df,
        TARGET_TABLE_NAME,
        target=TARGET_TARGET,
        schema=TARGET_SCHEMA,
    )
    display(target_dq_result["summary"])

    if not all(result["can_continue"] for result in (target_schema_result, target_dq_result)):
        raise RuntimeError("A target Guardrail blocked publication.")

    write_prep = write_pipeline_prep(
        transformed_df,
        read_prep,
        target=TARGET_TARGET,
    )

## TARGET 1 — Publish

The physical writer remains explicit. FabricOps records source progress only after the physical target write succeeds.

In [ ]:
if SHOULD_RUN:
    write_lakehouse_table(
        write_prep["df"],
        TARGET_TABLE_NAME,
        target=TARGET_TARGET,
        schema=TARGET_SCHEMA,
        mode=write_prep["mode"],
        options=write_prep["options"],
        load_strategy=write_prep["load_strategy"],
        load_strategy_parameters=write_prep["load_strategy_parameters"],
        processing_scope=write_prep["scope"],
        completion_context=write_prep["completion"],
    )

### Optional Warehouse target

Use the same prepared values with `write_warehouse_table()` when the configured target is a Warehouse. Warehouse supports governed `overwrite` and `append`; its current contract rejects `scd1` and `scd2` clearly rather than emulating MERGE logic in the notebook.

In [ ]:
# Replace the Lakehouse writer above with this block for a Warehouse target.
# if SHOULD_RUN:
#     write_warehouse_table(
#         write_prep["df"],
#         TARGET_SCHEMA,
#         TARGET_TABLE_NAME,
#         target=TARGET_TARGET,
#         mode=write_prep["mode"],
#         options=write_prep["options"],
#         load_strategy=write_prep["load_strategy"],
#         load_strategy_parameters=write_prep["load_strategy_parameters"],
#         completion_context=write_prep["completion"],
#     )

### Need another target?

Duplicate the TARGET block, rename its variables, and prepare and publish that target explicitly. Do not add loops or a notebook-level orchestration wrapper merely to hide the physical writes.

# Common patterns

| Source composition | Source strategy | Typical target strategy | Notes |
|---|---|---|---|
| Lakehouse Files → Lakehouse staging | `full_dataset` | `overwrite` | Use the matching file reader; no invented incremental file scope. |
| Warehouse table → Lakehouse | `incremental_watermark` | `append`, `scd1`, or `scd2` | Use the governed scope with `read_warehouse_table()`. |
| Multiple Lakehouse sources → transform → target | incremental strategies | workload-specific | Clone SOURCE blocks and define multi-source skip behaviour for the pipeline. |
| Multiple Lakehouse sources → transform → Lakehouse | `full_dataset` | `overwrite` | A straightforward rebuild when complete reads are appropriate. |

Target profiling is optional. Avoid reading back and profiling a large complete target after every incremental publication.